# Installation

## SWI-Prolog

SWI-Prolog is required to run LangPro. During the installation you will have to press ENTER to continue installing swi-prolog.

In [ ]:
#!sudo apt-get install software-properties-common
!sudo apt-add-repository -y ppa:swi-prolog/stable
!sudo apt-get update
!sudo apt-get install swi-prolog

In [ ]:
# test whether swi-prolog is installed and check its version
! swipl --version

## LangPro
Natural Tableau-based theorem prover that can operate on parsed sentences and detect semantic relations between a set of premises and a hypothesis. While creating this notebook, the `nl` branch of the LangPro repo is most up to date and stable.

In [ ]:
#! git clone https://github.com/kovvalsky/LangPro.git
#! git clone --single-branch --branch nl https://github.com/kovvalsky/LangPro.git
# just to make sure it points the specific commit on which the notebook was tested
#! cd LangPro; git reset --hard dfc0a00f46240e80675139089afdb82cab112332
#! cd LangPro; git pull
! git clone --single-branch --branch nl https://github.com/kovvalsky/LangPro.git

# EasyCCG

## C&C tools
C&C tools ([Clark&Curran, 2007](https://www.aclweb.org/anthology/J07-4004.pdf)) contain POS tagger, named-entity recognizer (NER), CCG parser, and Boxer. We need only the POS tagger, NER and parser. If the data doesn't contain named entities, the NER is irrelevant. It comes with only with the 🇬🇧 English models. As an input, LangPro requires the prolog format of the CCG derivation trees (hence, `--candc-printer boxer`).

In [ ]:
! candc/candc/bin/candc --version

In [ ]:
! tar -xzf candc/models/models-1.02.tgz -C candc/models

In [ ]:
# test that C&C tools (namely, POS tagger, NER and parser) are working
! echo "Parse this sentence for me" | ../candc/candc/bin/candc --models ../candc/models --candc-printer boxer

## Preparing Data for LangPro

LangPro requires two prolog files per dataset: a `*_sen.pl` file recording NLI problems with labels, and a `*_ccg.pl` file containing parse trees for all sentences. For each dataset in `datasets/`, we create those files under `langpro-datasets/` so parsing and theorem proving can be reused without repeated conversions.

The JSON datasets here contain premise sentences like `P1`, `P2`, etc., and a conclusion `C`. We write one `sen_id` entry per premise and one for the conclusion, then parse every sentence from the corresponding `.spl` file.

For each dataset file, create a tokenized sentence-per-line file and a LangPro prolog file. The parser expects tokenized input, so we use NLTK's TreebankWordTokenizer here, which avoids requiring downloaded punkt data.

In [ ]:
from nltk.tokenize import TreebankWordTokenizer
_tokenizer = TreebankWordTokenizer()

In [ ]:
import re
ENTITY_PATTERNS = [r'member of (.+? pathway)', r'(Gene .+?)\b']
def group_entities_cheat(stmnt):
    result = stmnt
    matches = []
    for pattern in ENTITY_PATTERNS:
        for m in re.finditer(pattern, stmnt):
            matches.append((m.start(1), m.end(1)))
    for start, end in sorted(matches, reverse=True):
        underscored = stmnt[start:end].strip().replace(" ", "_")
        result = result[:start] + underscored + result[end:]

    return result

In [ ]:
print(group_entities_cheat("Every member of ABC transporter disorders pathway is a member of Disorders of transmembrane transporters pathway."))
print(group_entities_cheat("Gene ABCD is a member of ABC transporter disorders pathway."))
print(group_entities_cheat("It is true that Gene ABCD is a member of Disorders of transmembrane transporters pathway."))


In [ ]:
import glob
import json
import os
from nltk.tokenize import TreebankWordTokenizer

def escape_prolog(text):
    return text.replace("'", r"\\'")

os.makedirs("langpro-datasets", exist_ok=True)
json_paths = sorted(glob.glob("../datasets/*.json"))
print(f"Found {len(json_paths)} dataset files in ../datasets/")

_tokenizer = TreebankWordTokenizer()

for json_path in json_paths:
    basename = os.path.splitext(os.path.basename(json_path))[0]
    sen_path = os.path.join("langpro-datasets", f"{basename}_sen.pl")
    spl_path = os.path.join("langpro-datasets", f"{basename}.spl")

    with open(json_path, encoding="utf-8") as f:
        problems = json.load(f)

    with open(sen_path, "w", encoding="utf-8") as sen_f, open(spl_path, "w", encoding="utf-8") as spl_f:
        for idx, problem in enumerate(problems, start=1):
            pid = f"{basename}_{idx}"

            premise_keys = sorted([k for k in problem.keys() if k.startswith('P')])
            premises = [problem[k] for k in premise_keys]
            premises = [group_entities_cheat(premise) for premise in premises]
            
            hypothesis = problem["C"]
            hypothesis = group_entities_cheat(hypothesis)


            # datasets are always generated as entailment in SylloBio-NLI framework, non-entailment is by mixing premises and conclusion at  eval time
            label = "yes"

            sen_f.write(f"% problem id = {pid}\n")

            num_premises = len(premises)
            for i, premise in enumerate(premises):
                sen_id = (num_premises + 1) * (idx - 1) + (i + 1)
                sen_f.write(f"sen_id({sen_id}, '{pid}', 'p', '{label}', '{escape_prolog(premise)}').\n")
                spl_f.write(" ".join(_tokenizer.tokenize(premise)) + "\n")

            hyp_id = (num_premises + 1) * idx
            sen_f.write(f"sen_id({hyp_id}, '{pid}', 'h', '{label}', '{escape_prolog(hypothesis)}').\n")
            spl_f.write(" ".join(_tokenizer.tokenize(hypothesis)) + "\n")

    print(f"Wrote {basename}_sen.pl and .spl")

Parsing the sentences with C&C tools. Stats and progress are written to `langpro-datasets/parsing.log` to avoid buffering large volumes of C&C output in the notebook. Already-parsed files are skipped so the cell is safe to re-run after a crash.

In [ ]:
import os
import glob
import subprocess
import tempfile
import time

spl_files = sorted(glob.glob("langpro-datasets/*.spl"))
print(f"Found {len(spl_files)} .spl files to parse.")

total_start = time.time()
count = 0

with open("langpro-datasets/parsing.log", "w") as log:
    for spl_path in spl_files:
        basename = os.path.basename(spl_path)
        output_path = os.path.join("langpro-datasets", basename.replace(".spl", "_easy_ccg.pl"))

        if os.path.exists(output_path):
            print(f"SKIP  {output_path} (already parsed)")
            continue
        
        with open(spl_path, "r") as f:
            input_text = f.read()

        # Temp file for prolog_to_boxer.py output
        with tempfile.NamedTemporaryFile(mode='w', suffix='.pl', delete=False) as tmp:
            tmp_path = tmp.name

        try:
            # Run pipeline to temp file
            easyccg_command = f"""candc/candc/bin/pos --model candc/models/models/pos | \
                candc/candc/bin/ner --model candc/models/models/ner -ofmt "%w|%p|%n \\n" | \
                java -jar easyccg/easyccg.jar --model easyccg/models/standard \
                -i POSandNERtagged -o prolog | \
                python3 LangPro/python/prolog_to_boxer.py > {tmp_path}"""
                
            t0 = time.time()
            result = subprocess.run(easyccg_command, input=input_text, shell=True, stderr=log, text=True, stdout=subprocess.DEVNULL)
            
            swipl_returncode = 1
            if result.returncode == 0:
                # Create combined file: converter + data
                with tempfile.NamedTemporaryFile(mode='w', suffix='.pl', delete=False) as combined:
                    combined_path = combined.name
                    # Write the converter first
                    with open("LangPro/prolog/converter/prologCCG_to_boxerCCG.pl", "r") as f:
                        combined.write(f.read())
                        combined.write("\n\n")
                    # Then append the data
                    with open(tmp_path, "r") as f:
                        combined.write(f.read())
                
                # Run SWI-Prolog with the combined file
                swipl_command = [
                    "swipl",
                    "-s", combined_path,
                    "-g", f"prolog_to_boxer('{output_path}')",
                    "-t", "halt"
                ]
                swipl_result = subprocess.run(swipl_command, stderr=log, stdout=subprocess.DEVNULL)
                swipl_returncode = swipl_result.returncode
                
                # Clean up combined file
                if os.path.exists(combined_path):
                    os.unlink(combined_path)

            elapsed = time.time() - t0
            status = "OK" if swipl_returncode == 0 else f"FAILED (exit {swipl_returncode})"
            count += 1
            print(f"[{count:>3}/{len(spl_files)}] {basename}: {status} ({elapsed:.1f}s)")

        finally:
            if os.path.exists(tmp_path):
                os.unlink(tmp_path)

total_elapsed = time.time() - total_start
mins, secs = divmod(int(total_elapsed), 60)
print(f"\nParsed {count} files in {mins}m {secs}s (avg {total_elapsed/count:.1f}s/file)" if count else "\nNothing to parse.")

Now we already have all necessary prolog files in `langpro-datasets/` for reasoning with LangPro.  
Note that if some sentence is not parsed, its corresponding `ccg($id,...` term won't be in `*_ccg.pl` file.

In [ ]:
! grep -cP "ccg\(\d+" langpro-datasets/*_cc_ccg.pl | wc -l

# NLI Proving

The elements of `parList` are described [here](https://github.com/kovvalsky/LangPro/wiki/Using-the-prover). The ones you might want to change are:
* `ral(50)` - rule application limit, which means that less you set there less time will be spend to find a proof and the results might be poor);
* `waif(filename)` - write answers in file. In case you want to have LangPro predictions in a file written.;
* `prprb` - by default LangPro prints problems that were not predicted correctly. This flag forces LangPro to print all the problems.

## Custom dataset proving

When running LangPro, we need to feed it with wordnet files to give it access to some lexical knowledge. Other files that needs to be given are prolog files with NLI problem descriptions and parses.

In [ ]:
# Directory where NLI proving judgements will be written
! mkdir -p results

In [ ]:
# Proving all NLI problems from the generated datasets.
# Adjust the glob pattern or loop over specific files as needed.
import glob
import os
import subprocess
import time

sen_files = sorted(glob.glob("langpro-datasets/*_sen.pl"))
print(f"Found {len(sen_files)} datasets to prove.")

gstart = time.time()
with open("proving.log", "w") as log:
    for sen_path in sen_files:
        start = time.time()
        basename = os.path.basename(sen_path).replace("_sen.pl", "")
        ccg_path = sen_path.replace("_sen.pl", "_easy_ccg.pl")
        result_path = os.path.join("results", f"{basename}_pred.txt")

        if not os.path.exists(ccg_path):
            print(f"SKIP {basename}: no CCG file found")
            continue

        cmd = [
            "swipl",
            "-g",
            f"parList([prprb, ral(50), aall, wn_ant, wn_sim, wn_der, constchk, waif('{result_path}')]), entail_all, halt",
            "-f",
            "LangPro/prolog/main.pl",
            "LangPro/WNProlog/wn.pl",
            sen_path,
            ccg_path,

        ]

        print(f"Proving {basename}...")
        subprocess.run(cmd, stderr=log, stdout=subprocess.DEVNULL)
        print(f"  -> written to {result_path}, took {time.time() - start:.1f}s")
total_elapsed = time.time() - gstart
print(f"\nProved {len(sen_files)} datasets in {total_elapsed:.1f}s (avg {total_elapsed/len(sen_files):.1f}s/dataset)")

# Preprocessing with scispacy NER

# Evaluating

In [ ]:
import glob
results_files = sorted(glob.glob("results/*_pred.txt"))
print(f"Accuracies for {len(results_files)} result files.")

for result_file in results_files:
    correct = 0
    total = 0
    with open(result_file, "r") as f:
        lines = f.readlines()
        lines = lines[3:]
        for line in lines:
            total += 1
            if "ENTAILMENT" in line:
                correct += 1
    accuracy = correct / total if total > 0 else 0
    print(f"{result_file.replace('results/', '').replace('_pred.txt', '').replace('-2-0', ''):<60}: {accuracy:>8.2f}")